In [1]:
import os, sys
from tqdm import tqdm
import torch
import numpy as np
from scipy import stats
from procrustes import rotational
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.abspath(''), '..'))
from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG, HARTREE2KCAL, AMU2ELECTRON_MASS

import openmm.app as app
from ase.io import read

In [2]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ff_path = os.path.join(os.path.abspath(''), '../scripts/ion_water_refit.xml')
ff = ForceFieldXML(ff_path, device=device)

In [24]:
f_water_box_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/condensed_phase_cmm/ion_water/boxes/f_water_216.pdb')

In [25]:
pdb = app.PDBFile(f_water_box_pdb_path)
top = Topology.fromOpenmm(pdb.topology, device)
system = ff.parametrize(top, use_fd_morse=True, cutoff_sr=5.0, use_lr_dispersion=True, use_hardness_change=False, use_polarization=True)

coords = torch.tensor(pdb.getPositions(asNumpy=True)._value / BOHR2NM, device=device, requires_grad=True)
box = torch.tensor([[vec.x / BOHR2NM, vec.y / BOHR2NM, vec.z / BOHR2NM] for vec in pdb.topology.getPeriodicBoxVectors()], device=device, requires_grad=True)

In [26]:
energies = system.getEnergy(coords, box)

In [27]:
print(energies)

{'bond': tensor(0.0019, grad_fn=<SumBackward0>), 'angle': tensor(0.0043, grad_fn=<SumBackward0>), 'torsion': tensor(0.), 'bond_bond': tensor(-0.0004, grad_fn=<SumBackward0>), 'bond_angle': tensor(-0.0004, grad_fn=<SumBackward0>), 'angle_angle': tensor(0.), 'torsion_bond': tensor(0.), 'torsion_angle': tensor(0.), 'torsion_angle_angle': tensor(0.), 'perm_elec': tensor(-5.4800, grad_fn=<AddBackward0>), 'pol': tensor(-4.4446, grad_fn=<DotBackward0>), 'ct_direct': tensor(-1.3720, grad_fn=<SumBackward0>), 'xpol': tensor(-0.1325, grad_fn=<SumBackward0>), 'pauli': tensor(5.5546, grad_fn=<SumBackward0>), 'disp': tensor(-1.9966, grad_fn=<AddBackward0>), 'total': tensor(-7.8656, grad_fn=<AddBackward0>)}


In [28]:
energies['total'].backward()
print(coords.grad)

tensor([[-0.0883, -0.1164,  0.2461],
        [-0.0042, -0.0111,  0.0054],
        [-0.0105,  0.0129, -0.0055],
        ...,
        [ 0.0067, -0.0014,  0.0047],
        [-0.0074, -0.0021, -0.0031],
        [-0.0008,  0.0010,  0.0284]])
